In [ ]:
# Cell 1: Environment + Drive Mount
!pip install transformers torch datasets scikit-learn \
             matplotlib seaborn numpy pandas tqdm \
             --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, os, pickle, time, warnings, shutil, re, unicodedata
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 150, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titlesize": 12,
    "axes.labelsize": 10, "font.family": "DejaVu Sans",
})
DISASTER_COLOR   = "#D85A30"
NODISASTER_COLOR = "#1D9E75"
NEUTRAL_COLOR    = "#7F77DD"

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from torch.optim import AdamW
from sklearn.metrics import (f1_score, classification_report,
                              confusion_matrix, precision_recall_curve,
                              average_precision_score, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split as sk_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("  No GPU detected — transformer training will be very slow")
    print("  Runtime → Change runtime type → T4 GPU")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_STAGE1_DIR = "/content/drive/MyDrive/Thesis_Stage1"
DRIVE_STAGE2_DIR = "/content/drive/MyDrive/Thesis_Stage2"
LOCAL_WORK_DIR   = "/content"
os.makedirs(DRIVE_STAGE2_DIR, exist_ok=True)

def save_to_drive(local_path, label=""):
    dest = os.path.join(DRIVE_STAGE2_DIR, os.path.basename(local_path))
    shutil.copy2(local_path, dest)
    print(f"  ✓ {os.path.basename(local_path)} → Drive  {label}")

print(f"\n  Stage 1 artifacts : {DRIVE_STAGE1_DIR}")
print(f"  Stage 2 outputs   : {DRIVE_STAGE2_DIR}")
print("\n✓ Cell 1 complete")


In [ ]:
# ===== Cell 2: Load Stage 1 Artifacts =====
# Loads df_clean.csv and split_indices.json from Drive.
# Reconstructs train/val/test DataFrames with identical splits to Stage 1.

def find_file(filename):
    """Search Stage 1 Drive folder, then local /content/."""
    candidates = [
        os.path.join(DRIVE_STAGE1_DIR, filename),
        os.path.join(LOCAL_WORK_DIR, filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

print("Loading Stage 1 artifacts …")

# df_clean.csv
df_path = find_file("df_clean.csv")
assert df_path, "df_clean.csv not found — check Drive/Thesis_Stage1/"
df = pd.read_csv(df_path)
print(f"  ✓ df_clean.csv : {len(df):,} rows  cols={list(df.columns)}")

# Ensure disaster_type column exists (Stage 1 new version adds it)
if "disaster_type" not in df.columns:
    # Fallback: re-derive from event column
    EVENT_TO_TYPE = {
        "earthquake":"earthquake","quake":"earthquake","flood":"flood",
        "floods":"flood","hurricane":"hurricane","cyclone":"hurricane",
        "typhoon":"hurricane","wildfire":"fire","fire":"fire",
        "explosion":"explosion","bombing":"explosion","shooting":"explosion",
    }
    def _infer(e):
        s = str(e).lower()
        for k,v in EVENT_TO_TYPE.items():
            if k in s: return v
        return "other"
    df["disaster_type"] = df.get("event", pd.Series(["unknown"]*len(df))).apply(_infer)
    df.loc[df["source"] == "humaid", "disaster_type"] = "other"
    print(f"  ⚠ disaster_type re-derived (old df_clean.csv)")

# split_indices.json
idx_path = find_file("split_indices.json")
assert idx_path, "split_indices.json not found"
with open(idx_path) as f:
    split_idx = json.load(f)

# Key name compatibility (train_idx or train)
train_key = "train_idx" if "train_idx" in split_idx else "train"
val_key   = "val_idx"   if "val_idx"   in split_idx else "val"
test_key  = "test_idx"  if "test_idx"  in split_idx else "test"

train_df = df.iloc[split_idx[train_key]].reset_index(drop=True)
val_df   = df.iloc[split_idx[val_key]].reset_index(drop=True)
test_df  = df.iloc[split_idx[test_key]].reset_index(drop=True)

print(f"  ✓ Splits: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print(f"    Disaster rate — train: {train_df['label_binary'].mean():.3f}  "
      f"test: {test_df['label_binary'].mean():.3f}")

# baseline_results.json
br_path = find_file("baseline_results.json")
baseline_results = None
if br_path:
    with open(br_path) as f:
        baseline_results = json.load(f)
    print(f"  ✓ Baseline LR macro F1  : {baseline_results['logistic_regression']['macro_f1']:.4f}")
    print(f"  ✓ Baseline FT macro F1  : {baseline_results['fasttext_proxy']['macro_f1']:.4f}")

print("\n✓ Cell 2 complete")

In [ ]:
# Cell 3: Dataset Class + Evaluate Helper

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }


def evaluate(model, loader):
    """Run model on a DataLoader, return dict with preds + metrics."""
    model.eval()
    all_preds, all_proba, all_labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            labs  = batch["label"].to(DEVICE)
            logits = model(input_ids=ids, attention_mask=mask).logits
            proba  = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds  = (proba >= 0.5).astype(int)
            all_preds.extend(preds)
            all_proba.extend(proba)
            all_labels.extend(labs.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_proba)

    return {
        "y_pred"     : y_pred,
        "y_proba"    : y_prob,
        "binary_f1"  : round(float(f1_score(y_true, y_pred, average="binary",   zero_division=0)), 4),
        "macro_f1"   : round(float(f1_score(y_true, y_pred, average="macro",    zero_division=0)), 4),
        "weighted_f1": round(float(f1_score(y_true, y_pred, average="weighted", zero_division=0)), 4),
    }


def predict(model, tokenizer, texts, batch_size=128, threshold=0.5):
    """Batch-classify raw text strings. Returns (preds, probas)."""
    model.eval()
    all_proba = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, padding=True, truncation=True,
                          max_length=128, return_tensors="pt")
        enc   = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            proba = torch.softmax(
                model(**enc).logits, dim=-1)[:, 1].cpu().numpy()
        all_proba.extend(proba)
    all_proba = np.array(all_proba)
    return (all_proba >= threshold).astype(int), all_proba

print("✓ Cell 3 complete — TweetDataset, evaluate(), predict() defined")

In [ ]:
# ===== Cell 4: Training Function (Speed-Optimised, Drive-Persistent) =====

def train_model(model_name, train_texts, train_labels,
                val_texts, val_labels, checkpoint_name,
                n_epochs=4, batch_size=64, grad_accum=2,
                lr=2e-5, warmup_ratio=0.1, max_len=128,
                patience=3, use_fp16=True):
    """
    Fine-tune a HuggingFace sequence classifier.

    Saves a checkpoint after every epoch. Auto-resumes from the last
    saved checkpoint if the cell is re-run after a session crash.

    Returns: (model, tokenizer, history)
    """
    ckpt_dir = os.path.join(DRIVE_STAGE2_DIR, checkpoint_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    state_file = os.path.join(ckpt_dir, "train_state.json")
    best_dir   = os.path.join(ckpt_dir, "best_model")

    # Load or initialise model
    start_epoch = 0
    history     = []
    best_val_f1 = 0.0
    no_improve  = 0

    # Check for existing checkpoint to resume from
    if os.path.exists(state_file):
        with open(state_file) as f:
            state = json.load(f)
        start_epoch = state.get("epoch", 0)
        history     = state.get("history", [])
        best_val_f1 = state.get("best_val_f1", 0.0)
        no_improve  = state.get("no_improve", 0)
        resume_from = state.get("last_ckpt_path", model_name)
        print(f"  ↩ Resuming from epoch {start_epoch}  (best val macro F1={best_val_f1:.4f})")
        tokenizer = AutoTokenizer.from_pretrained(resume_from)
        model     = AutoModelForSequenceClassification.from_pretrained(
            resume_from, num_labels=2)
    else:
        print(f"  Starting fresh from {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model     = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2)

    model = model.to(DEVICE)

    # Try torch.compile for extra speed (PyTorch ≥ 2.0 only)
    try:
        model = torch.compile(model)
        print("  ✓ torch.compile() applied")
    except Exception:
        pass

    # DataLoaders
    train_ds = TweetDataset(train_texts, train_labels, tokenizer, max_len)
    val_ds   = TweetDataset(val_texts,   val_labels,   tokenizer, max_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=4,
                              pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_ds,   batch_size=128,
                              shuffle=False, num_workers=4,
                              pin_memory=True, persistent_workers=True)

    # Optimiser + scheduler
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps   = (len(train_loader) // grad_accum) * n_epochs
    warmup_steps  = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps)

    scaler = torch.cuda.amp.GradScaler(enabled=(use_fp16 and DEVICE.type == "cuda"))

    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"  Train: {len(train_ds):,}  Val: {len(val_ds):,}")
    print(f"  Epochs: {n_epochs}  Batch: {batch_size}  GradAccum: {grad_accum}")
    print(f"  Effective batch: {batch_size*grad_accum}  LR: {lr}  FP16: {use_fp16}")
    print(f"  Checkpoint dir: {ckpt_dir}")
    print(f"{'='*60}")

    if start_epoch >= n_epochs:
        print(f"  All {n_epochs} epochs already done — loading best model")
        model     = AutoModelForSequenceClassification.from_pretrained(
            best_dir, num_labels=2).to(DEVICE)
        tokenizer = AutoTokenizer.from_pretrained(best_dir)
        return model, tokenizer, history

    # Training loop
    for epoch in range(start_epoch, n_epochs):
        model.train()
        total_loss   = 0.0
        n_batches    = 0
        optimizer.zero_grad()
        t0 = time.time()

        for step, batch in enumerate(train_loader):
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            labs  = batch["label"].to(DEVICE)

            with torch.cuda.amp.autocast(
                    enabled=(use_fp16 and DEVICE.type == "cuda")):
                loss = model(input_ids=ids, attention_mask=mask,
                              labels=labs).loss / grad_accum

            scaler.scale(loss).backward()

            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += loss.item() * grad_accum
            n_batches  += 1

        avg_loss = total_loss / n_batches
        val_res  = evaluate(model, val_loader)
        elapsed  = int(time.time() - t0)

        print(f"  Epoch {epoch+1}/{n_epochs} | "
              f"train_loss={avg_loss:.4f} | "
              f"val_macro_f1={val_res['macro_f1']:.4f} | "
              f"val_binary_f1={val_res['binary_f1']:.4f} | {elapsed}s")

        # Save per-epoch checkpoint to Drive
        ep_ckpt = os.path.join(ckpt_dir, f"epoch_{epoch+1}")
        model._orig_mod.save_pretrained(ep_ckpt) \
            if hasattr(model, "_orig_mod") else model.save_pretrained(ep_ckpt)
        tokenizer.save_pretrained(ep_ckpt)
        print(f"    ✓ Epoch {epoch+1} checkpoint saved → Drive")

        # Track best model
        if val_res["macro_f1"] > best_val_f1:
            best_val_f1 = val_res["macro_f1"]
            no_improve  = 0
            os.makedirs(best_dir, exist_ok=True)
            model._orig_mod.save_pretrained(best_dir) \
                if hasattr(model, "_orig_mod") else model.save_pretrained(best_dir)
            tokenizer.save_pretrained(best_dir)
            print(f"    ✓ New best val macro F1: {best_val_f1:.4f} → saved as best_model")
        else:
            no_improve += 1
            print(f"    No improvement ({no_improve}/{patience})")

        history.append({
            "epoch"        : epoch + 1,
            "train_loss"   : round(avg_loss, 4),
            "val_macro_f1" : round(val_res["macro_f1"], 4),
            "val_binary_f1": round(val_res["binary_f1"], 4),
            "elapsed_s"    : elapsed,
        })

        # Save training state for resumption
        with open(state_file, "w") as f:
            json.dump({
                "epoch"          : epoch + 1,
                "history"        : history,
                "best_val_f1"    : best_val_f1,
                "no_improve"     : no_improve,
                "last_ckpt_path" : ep_ckpt,
            }, f, indent=2)

        # Early stopping
        if no_improve >= patience:
            print(f"  Early stopping — no improvement for {patience} epochs")
            break

    print(f"\n  Best val macro F1: {best_val_f1:.4f}")

    # Load best model for return
    model     = AutoModelForSequenceClassification.from_pretrained(
        best_dir, num_labels=2).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(best_dir)
    return model, tokenizer, history

print("✓ Cell 4 complete — train_model() defined")

In [ ]:
# ===== Cell 5: Fine-tune DistilBERT =====

print("Starting DistilBERT fine-tuning …")
print("Checkpoints saved to Drive after every epoch.\n")

distilbert_model, distilbert_tokenizer, distilbert_history = train_model(
    model_name      = "distilbert-base-uncased",
    train_texts     = train_df["clean_text"].tolist(),
    train_labels    = train_df["label_binary"].tolist(),
    val_texts       = val_df["clean_text"].tolist(),
    val_labels      = val_df["label_binary"].tolist(),
    checkpoint_name = "distilbert_checkpoints",
    n_epochs        = 4,
    batch_size      = 64,
    grad_accum      = 2,
    lr              = 2e-5,
    warmup_ratio    = 0.1,
    patience        = 3,
    use_fp16        = True,
)

# Save final model to Drive
final_db_dir = os.path.join(DRIVE_STAGE2_DIR, "distilbert_model")
distilbert_model.save_pretrained(final_db_dir)
distilbert_tokenizer.save_pretrained(final_db_dir)
print(f"\n  ✓ Saved to Drive: {final_db_dir}")

print(f"\nDistilBERT training history:")
for h in distilbert_history:
    print(f"  Epoch {h['epoch']}: train_loss={h['train_loss']} | "
          f"val_macro_f1={h['val_macro_f1']} | val_binary_f1={h['val_binary_f1']} | {h['elapsed_s']}s")

print("\n✓ Cell 5 complete — DistilBERT trained and saved to Drive")

In [ ]:
# ===== Cell 6: Fine-tune RoBERTa (optional) =====

RUN_ROBERTA = True

ROBERTA_AVAILABLE = False
roberta_model     = None
roberta_tokenizer = None
roberta_history   = []

if RUN_ROBERTA:
    print("Starting RoBERTa fine-tuning …")
    print("⏳ Expected: ~18–22 min/epoch × 4 epochs = ~80 min total\n")

    roberta_model, roberta_tokenizer, roberta_history = train_model(
        model_name      = "cardiffnlp/twitter-roberta-base",
        train_texts     = train_df["clean_text"].tolist(),
        train_labels    = train_df["label_binary"].tolist(),
        val_texts       = val_df["clean_text"].tolist(),
        val_labels      = val_df["label_binary"].tolist(),
        checkpoint_name = "roberta_checkpoints",
        n_epochs        = 4,
        batch_size      = 32,
        grad_accum      = 4,
        lr              = 2e-5,
        warmup_ratio    = 0.1,
        patience        = 3,
        use_fp16        = True,
    )
    final_rb_dir = os.path.join(DRIVE_STAGE2_DIR, "roberta_model")
    roberta_model.save_pretrained(final_rb_dir)
    roberta_tokenizer.save_pretrained(final_rb_dir)
    print(f"\n  ✓ Saved to Drive: {final_rb_dir}")
    ROBERTA_AVAILABLE = True

else:
    print("RUN_ROBERTA = False — checking for existing checkpoint on Drive …")
    rb_ckpt = os.path.join(DRIVE_STAGE2_DIR, "roberta_model")
    if os.path.exists(rb_ckpt) and os.listdir(rb_ckpt):
        roberta_model     = AutoModelForSequenceClassification.from_pretrained(
            rb_ckpt, num_labels=2).to(DEVICE)
        roberta_tokenizer = AutoTokenizer.from_pretrained(rb_ckpt)
        ROBERTA_AVAILABLE = True
        print(f"  ✓ RoBERTa loaded from Drive: {rb_ckpt}")
        rb_state = os.path.join(DRIVE_STAGE2_DIR, "roberta_checkpoints", "train_state.json")
        if os.path.exists(rb_state):
            with open(rb_state) as f:
                roberta_history = json.load(f).get("history", [])
    else:
        print("  No RoBERTa checkpoint found — downstream cells use DistilBERT only")

print("\n✓ Cell 6 complete")

In [ ]:
# ===== Cell 7: Evaluate on Test Set =====
# Evaluate both models at default threshold=0.5
# Threshold is calibrated in Cell 8 — these are the uncalibrated results

print("Evaluating on test set …")

# Build test DataLoaders
test_loader_db = DataLoader(
    TweetDataset(test_df["clean_text"].tolist(),
                 test_df["label_binary"].tolist(),
                 distilbert_tokenizer, max_len=128),
    batch_size=128, shuffle=False, num_workers=4
)

db_test     = evaluate(distilbert_model, test_loader_db)
y_test_true = test_df["label_binary"].values

models_to_eval = [("DistilBERT", db_test)]

if ROBERTA_AVAILABLE:
    test_loader_rb = DataLoader(
        TweetDataset(test_df["clean_text"].tolist(),
                     test_df["label_binary"].tolist(),
                     roberta_tokenizer, max_len=128),
        batch_size=64, shuffle=False, num_workers=4
    )
    rb_test = evaluate(roberta_model, test_loader_rb)
    models_to_eval.append(("RoBERTa-twitter", rb_test))

for name, res in models_to_eval:
    print(f"\n{'='*55}")
    print(f"{name} — Test Set Results (threshold=0.5)")
    print(f"{'='*55}")
    print(f"  Binary F1   : {res['binary_f1']:.4f}")
    print(f"  Macro F1    : {res['macro_f1']:.4f}")
    print(f"  Weighted F1 : {res['weighted_f1']:.4f}")
    print(f"\n{classification_report(y_test_true, res['y_pred'], target_names=['Non-disaster','Disaster'])}")

# Figure 8: Training curves
n_plots = 2 if (ROBERTA_AVAILABLE and roberta_history) else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7*n_plots, 4))
if n_plots == 1: axes = [axes]

plot_data = [(distilbert_history, "DistilBERT", DISASTER_COLOR)]
if ROBERTA_AVAILABLE and roberta_history:
    plot_data.append((roberta_history, "RoBERTa-twitter", NEUTRAL_COLOR))

for ax, (history, name, color) in zip(axes, plot_data):
    epochs     = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_f1     = [h["val_macro_f1"] for h in history]
    ax2 = ax.twinx()
    ax.plot(epochs, train_loss, color=color,      lw=2, marker="o", label="Train loss")
    ax2.plot(epochs, val_f1,   color="steelblue", lw=2, marker="s", ls="--", label="Val macro F1")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Train Loss", color=color)
    ax2.set_ylabel("Val Macro F1", color="steelblue")
    ax.set_title(name)
    ax.legend(loc="upper left"); ax2.legend(loc="lower right")

fig.suptitle("Figure 8 — Transformer Training Curves", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("/content/fig8_training_curves.png", bbox_inches="tight", dpi=150)
save_to_drive("/content/fig8_training_curves.png")
plt.show(); print("✓ Figure 8 saved")

# Figure 9: Confusion matrices
fig, axes = plt.subplots(1, len(models_to_eval),
                          figsize=(6*len(models_to_eval), 5))
if len(models_to_eval) == 1: axes = [axes]
for ax, (name, res) in zip(axes, models_to_eval):
    cm = confusion_matrix(y_test_true, res["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Non-disaster","Disaster"])\
        .plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=11)
fig.suptitle("Figure 9 — Transformer Confusion Matrices (Test Set)", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("/content/fig9_confusion_matrices.png", bbox_inches="tight", dpi=150)
save_to_drive("/content/fig9_confusion_matrices.png")
plt.show(); print("✓ Figure 9 saved")

print("\n✓ Cell 7 complete")

In [ ]:
# Cell 8: Threshold Calibration

print("Computing validation probabilities for threshold calibration …")

val_loader_db = DataLoader(
    TweetDataset(val_df["clean_text"].tolist(),
                 val_df["label_binary"].tolist(),
                 distilbert_tokenizer, max_len=128),
    batch_size=128, shuffle=False, num_workers=4
)
db_val     = evaluate(distilbert_model, val_loader_db)
y_val_true = val_df["label_binary"].values

THRESHOLDS = np.arange(0.30, 0.91, 0.05)

def sweep_thresholds(y_true, y_proba, thresholds):
    rows = []
    for t in thresholds:
        yp = (y_proba >= t).astype(int)
        rows.append({
            "threshold": round(float(t), 2),
            "macro_f1" : round(float(f1_score(y_true, yp, average="macro",  zero_division=0)), 6),
            "binary_f1": round(float(f1_score(y_true, yp, average="binary", zero_division=0)), 6),
        })
    return pd.DataFrame(rows)

db_thresh_df = sweep_thresholds(y_val_true, db_val["y_proba"], THRESHOLDS)
db_best      = db_thresh_df.loc[db_thresh_df["macro_f1"].idxmax()]
DISTILBERT_THRESHOLD = float(db_best["threshold"])

print(f"\nDistilBERT best threshold: {DISTILBERT_THRESHOLD:.2f}  "
      f"(val macro F1={db_best['macro_f1']:.4f})")
print(f"\nThreshold sweep:")
print(db_thresh_df[["threshold","macro_f1","binary_f1"]].to_string(index=False))

ROBERTA_THRESHOLD = 0.5
if ROBERTA_AVAILABLE:
    val_loader_rb = DataLoader(
        TweetDataset(val_df["clean_text"].tolist(),
                     val_df["label_binary"].tolist(),
                     roberta_tokenizer, max_len=128),
        batch_size=64, shuffle=False, num_workers=4
    )
    rb_val        = evaluate(roberta_model, val_loader_rb)
    rb_thresh_df  = sweep_thresholds(y_val_true, rb_val["y_proba"], THRESHOLDS)
    rb_best       = rb_thresh_df.loc[rb_thresh_df["macro_f1"].idxmax()]
    ROBERTA_THRESHOLD = float(rb_best["threshold"])
    print(f"\nRoBERTa best threshold : {ROBERTA_THRESHOLD:.2f}  "
          f"(val macro F1={rb_best['macro_f1']:.4f})")

# Figure 10: Threshold calibration curves
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(db_thresh_df["threshold"], db_thresh_df["macro_f1"],
        color=DISASTER_COLOR, lw=2, marker="o", label="DistilBERT")
if ROBERTA_AVAILABLE:
    ax.plot(rb_thresh_df["threshold"], rb_thresh_df["macro_f1"],
            color=NEUTRAL_COLOR, lw=2, marker="s", ls="--", label="RoBERTa-twitter")
ax.axvline(DISTILBERT_THRESHOLD, color=DISASTER_COLOR, ls=":", alpha=0.7,
           label=f"DistilBERT best = {DISTILBERT_THRESHOLD:.2f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Val Macro F1")
ax.set_title("Figure 10 — Threshold Calibration (Validation Set)")
ax.legend(); ax.set_xlim([0.28, 0.92])
plt.tight_layout()
plt.savefig("/content/fig10_threshold_calibration.png", bbox_inches="tight", dpi=150)
save_to_drive("/content/fig10_threshold_calibration.png")
plt.show(); print("✓ Figure 10 saved")

# Figure 11: PR curves on test set
fig, ax = plt.subplots(figsize=(8, 6))
random_ap = float(y_test_true.mean())
for res, name, color, ls in [
    (db_test, "DistilBERT",      DISASTER_COLOR, "-"),
    *( [(rb_test, "RoBERTa-twitter", NEUTRAL_COLOR, "--")]
       if ROBERTA_AVAILABLE else [] ),
]:
    prec, rec, _ = precision_recall_curve(y_test_true, res["y_proba"])
    ap = average_precision_score(y_test_true, res["y_proba"])
    ax.plot(rec, prec, color=color, lw=2, ls=ls,
            label=f"{name} (AP={ap:.3f})")
ax.axhline(random_ap, color="grey", ls=":", lw=1.5,
           label=f"Random (AP={random_ap:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Figure 11 — Precision-Recall Curves (Test Set)")
ax.legend(fontsize=9); ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
plt.tight_layout()
plt.savefig("/content/fig11_pr_curves.png", bbox_inches="tight", dpi=150)
save_to_drive("/content/fig11_pr_curves.png")
plt.show(); print("✓ Figure 11 saved")

print("\n✓ Cell 8 complete")


In [ ]:
# Cell 9: Full Comparison Table
# Thesis Table 1: all 4 models at calibrated thresholds

db_pred_cal = (db_test["y_proba"] >= DISTILBERT_THRESHOLD).astype(int)
db_bin_cal  = f1_score(y_test_true, db_pred_cal, average="binary")
db_mac_cal  = f1_score(y_test_true, db_pred_cal, average="macro")
db_wt_cal   = f1_score(y_test_true, db_pred_cal, average="weighted")

print("=" * 72)
print("THESIS TABLE 1 — Full Model Comparison (Test Set, calibrated threshold)")
print("=" * 72)
print(f"\n{'Model':<35} {'Binary F1':>10} {'Macro F1':>10} "
      f"{'Weighted F1':>12} {'Threshold':>10}")
print("-" * 72)

table_rows = []
if baseline_results:
    table_rows += [
        ("LR (TF-IDF) [baseline]",
         baseline_results["logistic_regression"]["binary_f1"],
         baseline_results["logistic_regression"]["macro_f1"],
         baseline_results["logistic_regression"]["weighted_f1"], 0.50),
        ("FastText-proxy [baseline]",
         baseline_results["fasttext_proxy"]["binary_f1"],
         baseline_results["fasttext_proxy"]["macro_f1"],
         baseline_results["fasttext_proxy"]["weighted_f1"], 0.50),
    ]

table_rows.append((
    "DistilBERT (fine-tuned)",
    round(db_bin_cal, 4), round(db_mac_cal, 4),
    round(db_wt_cal, 4),  DISTILBERT_THRESHOLD
))

if ROBERTA_AVAILABLE:
    rb_pred_cal = (rb_test["y_proba"] >= ROBERTA_THRESHOLD).astype(int)
    table_rows.append((
        "RoBERTa-twitter (fine-tuned)",
        round(f1_score(y_test_true, rb_pred_cal, average="binary"),   4),
        round(f1_score(y_test_true, rb_pred_cal, average="macro"),    4),
        round(f1_score(y_test_true, rb_pred_cal, average="weighted"), 4),
        ROBERTA_THRESHOLD
    ))

for row in table_rows:
    print(f"{row[0]:<35} {row[1]:>10.4f} {row[2]:>10.4f} {row[3]:>12.4f} {row[4]:>10.2f}")

print(f"\n  RQ1 target: Binary F1 ≥ 0.85  |  Macro F1 ≥ 0.75")
db_pass = "✅ PASS" if round(db_bin_cal,4) >= 0.85 else "❌ FAIL"
print(f"  DistilBERT: {db_pass}")

# Figure 12: Full comparison bar chart
names_plot  = [r[0].replace(" [baseline]","").replace(" (fine-tuned)","")
               for r in table_rows]
binary_vals = [r[1] for r in table_rows]
macro_vals  = [r[2] for r in table_rows]
colors_plot = ([NODISASTER_COLOR]*2 + [DISASTER_COLOR] +
               ([NEUTRAL_COLOR] if ROBERTA_AVAILABLE else []))

x = np.arange(len(names_plot)); w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x-w/2, binary_vals, w, label="Binary F1", color=colors_plot, alpha=0.85)
b2 = ax.bar(x+w/2, macro_vals,  w, label="Macro F1",  color=colors_plot, alpha=0.55)
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
ax.axhline(0.85, color="grey", ls="--", lw=1.2, alpha=0.7, label="Binary F1 target (0.85)")
ax.axhline(0.75, color="grey", ls=":",  lw=1.0, alpha=0.7, label="Macro F1 target (0.75)")
ax.set_xticks(x); ax.set_xticklabels(names_plot, fontsize=8)
ax.set_ylim(0, 1.05); ax.set_ylabel("F1 Score")
ax.set_title("Figure 12 — Full Model Comparison (Test Set, calibrated threshold)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("/content/fig12_full_comparison.png", bbox_inches="tight", dpi=150)
save_to_drive("/content/fig12_full_comparison.png")
plt.show(); print("✓ Figure 12 saved")

print("\n✓ Cell 9 complete")

In [ ]:
# ===== Cell 10: LOTO Evaluation =====
# Leave-One-Type-Out: train on 5 types, test on 1 type.
# Tests whether the model generalises to unseen disaster categories.

LOTO_TYPES      = ["earthquake", "flood", "fire", "hurricane", "explosion", "other"]
LOTO_STATE_FILE = os.path.join(DRIVE_STAGE2_DIR, "loto_results.json")

# Load existing results (resume after crash)
if os.path.exists(LOTO_STATE_FILE):
    with open(LOTO_STATE_FILE) as f:
        loto_results = json.load(f)
    done_types = {r["held_out_type"] for r in loto_results}
    remaining  = [t for t in LOTO_TYPES if t not in done_types]
    print(f"✓ Resuming LOTO — completed: {sorted(done_types)}")
    print(f"  Remaining: {remaining}  (~{len(remaining)*30} min)")
else:
    loto_results = []
    done_types   = set()
    print(f"Starting LOTO from scratch — 6 types to evaluate (~3 hours total)")

print(f"\nDisaster type distribution:")
for dtype, count in df["disaster_type"].value_counts().items():
    warn = " ⚠ SMALL (high variance expected)" if count < 5000 else ""
    print(f"  {dtype:<15}: {count:>8,}  ({count/len(df)*100:.1f}%){warn}")

for held_out_type in LOTO_TYPES:
    if held_out_type in done_types:
        print(f"\n  ✓ Skipping '{held_out_type}' — already in Drive")
        continue

    print(f"\n{'='*60}")
    print(f"LOTO run: held out = '{held_out_type}'")
    print(f"{'='*60}")

    loto_train = df[df["disaster_type"] != held_out_type].reset_index(drop=True)
    loto_test  = df[df["disaster_type"] == held_out_type].reset_index(drop=True)

    if len(loto_test) < 100:
        print(f"  ⚠ Only {len(loto_test)} test rows — skipping")
        loto_results.append({
            "held_out_type": held_out_type, "n_test": len(loto_test),
            "binary_f1": None, "macro_f1": None, "note": "skipped_too_few"
        })
        done_types.add(held_out_type)
        with open(LOTO_STATE_FILE, "w") as f:
            json.dump(loto_results, f, indent=2)
        continue

    # 10% of training set as validation
    loto_tr, loto_val = sk_split(
        loto_train, test_size=0.10, random_state=42,
        stratify=loto_train["label_binary"]
    )
    loto_tr  = loto_tr.reset_index(drop=True)
    loto_val = loto_val.reset_index(drop=True)

    print(f"  Train: {len(loto_tr):,}  Val: {len(loto_val):,}  Test: {len(loto_test):,}")
    print(f"  Test disaster rate: {loto_test['label_binary'].mean():.3f}")

    t_start = time.time()
    loto_model, loto_tok, _ = train_model(
        model_name      = "distilbert-base-uncased",
        train_texts     = loto_tr["clean_text"].tolist(),
        train_labels    = loto_tr["label_binary"].tolist(),
        val_texts       = loto_val["clean_text"].tolist(),
        val_labels      = loto_val["label_binary"].tolist(),
        checkpoint_name = f"loto_{held_out_type}_checkpoints",
        n_epochs        = 2,
        batch_size      = 64,
        grad_accum      = 2,
        lr              = 2e-5,
        patience        = 2,
        use_fp16        = True,
    )

    # Evaluate on held-out type
    loto_test_ds  = TweetDataset(loto_test["clean_text"].tolist(),
                                  loto_test["label_binary"].tolist(),
                                  loto_tok, max_len=128)
    loto_test_loader = DataLoader(loto_test_ds, batch_size=128,
                                   shuffle=False, num_workers=4)
    loto_eval = evaluate(loto_model, loto_test_loader)
    elapsed   = round((time.time()-t_start)/60, 1)

    row = {
        "held_out_type"      : held_out_type,
        "n_train"            : len(loto_tr),
        "n_val"              : len(loto_val),
        "n_test"             : len(loto_test),
        "binary_f1"          : loto_eval["binary_f1"],
        "macro_f1"           : loto_eval["macro_f1"],
        "weighted_f1"        : loto_eval["weighted_f1"],
        "disaster_rate_test" : round(float(loto_test["label_binary"].mean()), 3),
        "elapsed_min"        : elapsed,
        "note"               : "ok",
    }
    loto_results.append(row)
    done_types.add(held_out_type)

    print(f"\n  ✓ Result: binary_f1={row['binary_f1']}  macro_f1={row['macro_f1']}  ({elapsed}min)")
    print(f"\n{classification_report(loto_test['label_binary'].values, loto_eval['y_pred'], target_names=['Non-disaster','Disaster'], digits=4)}")

    # Save to Drive immediately
    with open(LOTO_STATE_FILE, "w") as f:
        json.dump(loto_results, f, indent=2)
    print(f"  ✓ loto_results.json saved to Drive")

    del loto_model
    torch.cuda.empty_cache()

# LOTO Summary + Figure 13
valid_loto = [r for r in loto_results if r.get("macro_f1") is not None]
loto_df    = pd.DataFrame(valid_loto)

if len(loto_df) > 0:
    print("\n" + "=" * 65)
    print("LOTO RESULTS — DistilBERT (Leave-One-Type-Out)")
    print("=" * 65)
    print(f"\n{'Type':<15} {'N_test':>8} {'Binary F1':>10} {'Macro F1':>10} "
          f"{'Disaster%':>10} {'Time(min)':>10}")
    print("-" * 65)
    for _, row in loto_df.iterrows():
        flag = " ⚠" if row.get("n_test", 9999) < 2000 else ""
        print(f"{row['held_out_type']:<15} {int(row['n_test']):>8,} "
              f"{row['binary_f1']:>10.4f} {row['macro_f1']:>10.4f} "
              f"{row['disaster_rate_test']*100:>9.1f}% "
              f"{row.get('elapsed_min',0):>9.1f}m{flag}")

    mean_mac = loto_df["macro_f1"].mean()
    min_row  = loto_df.loc[loto_df["macro_f1"].idxmin()]
    max_row  = loto_df.loc[loto_df["macro_f1"].idxmax()]

    print(f"\n  Mean macro F1  : {mean_mac:.4f}")
    print(f"  Best type      : {max_row['held_out_type']} ({max_row['macro_f1']:.4f})")
    print(f"  Hardest type   : {min_row['held_out_type']} ({min_row['macro_f1']:.4f})")
    print(f"  Types ≥ 0.75   : {(loto_df['macro_f1'] >= 0.75).sum()}/{len(loto_df)}")

    # Figure 13
    colors_l = [DISASTER_COLOR if f>=0.80 else
                NEUTRAL_COLOR  if f>=0.75 else
                NODISASTER_COLOR for f in loto_df["macro_f1"]]
    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(loto_df["held_out_type"], loto_df["macro_f1"],
                  color=colors_l, edgecolor="white", width=0.55, alpha=0.88)
    for bar, val, n in zip(bars, loto_df["macro_f1"], loto_df["n_test"]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{val:.3f}\n(n={int(n):,})",
                ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.axhline(mean_mac, color="grey", ls="--", lw=1.5,
               label=f"Mean = {mean_mac:.3f}")
    ax.axhline(0.75, color=NEUTRAL_COLOR, ls=":", lw=1.5,
               label="Target (0.75)")
    ax.set_xlabel("Held-out Disaster Type"); ax.set_ylabel("Macro F1")
    ax.set_title("Figure 13 — LOTO Evaluation: DistilBERT Generalisation\n"
                 "(⚠ = small test set, higher variance)")
    ax.set_ylim(0, 1.10); ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("/content/fig13_loto_results.png", bbox_inches="tight", dpi=150)
    save_to_drive("/content/fig13_loto_results.png")
    plt.show(); print("✓ Figure 13 saved")

remaining = [t for t in LOTO_TYPES if t not in done_types]
if remaining:
    print(f"\n⚠ Remaining: {remaining} — re-run this cell to continue")
else:
    print(f"\n✓ All 6 LOTO types complete")

print("\n✓ Cell 10 complete")


In [ ]:
# Cell 11: Save All Stage 2 Artifacts

print("=" * 55)
print("Saving Stage 2 artifacts to Drive")
print("=" * 55)

# Transformer results JSON
transformer_results = {
    "distilbert": {
        "binary_f1"  : round(db_bin_cal, 4),
        "macro_f1"   : round(db_mac_cal, 4),
        "weighted_f1": round(db_wt_cal,  4),
        "threshold"  : DISTILBERT_THRESHOLD,
        "history"    : distilbert_history,
    }
}
if ROBERTA_AVAILABLE:
    rb_pred_cal_final = (rb_test["y_proba"] >= ROBERTA_THRESHOLD).astype(int)
    transformer_results["roberta"] = {
        "binary_f1"  : round(f1_score(y_test_true, rb_pred_cal_final, average="binary"),   4),
        "macro_f1"   : round(f1_score(y_test_true, rb_pred_cal_final, average="macro"),    4),
        "weighted_f1": round(f1_score(y_test_true, rb_pred_cal_final, average="weighted"), 4),
        "threshold"  : ROBERTA_THRESHOLD,
        "history"    : roberta_history,
    }

with open("/content/transformer_results.json", "w") as f:
    json.dump(transformer_results, f, indent=2)
save_to_drive("/content/transformer_results.json")

# LOTO results
if os.path.exists(LOTO_STATE_FILE):
    print(f"  ✓ loto_results.json already on Drive")

# Figures
for fig_file in [
    "/content/fig8_training_curves.png",
    "/content/fig9_confusion_matrices.png",
    "/content/fig10_threshold_calibration.png",
    "/content/fig11_pr_curves.png",
    "/content/fig12_full_comparison.png",
    "/content/fig13_loto_results.png",
]:
    if os.path.exists(fig_file):
        save_to_drive(fig_file)

# Final summary
print(f"""
{'='*65}
STAGE 2 COMPLETE
{'='*65}

Transformer results (test set, calibrated threshold):
  DistilBERT  : Binary={transformer_results['distilbert']['binary_f1']:.4f}  \
Macro={transformer_results['distilbert']['macro_f1']:.4f}  \
@t={DISTILBERT_THRESHOLD:.2f}""" +
(f"""
  RoBERTa     : Binary={transformer_results['roberta']['binary_f1']:.4f}  \
Macro={transformer_results['roberta']['macro_f1']:.4f}  \
@t={ROBERTA_THRESHOLD:.2f}""" if ROBERTA_AVAILABLE else """
  RoBERTa     : not run (set RUN_ROBERTA=True)""") +
f"""

Primary model for Stage 3: DistilBERT
  Saved at    : {DRIVE_STAGE2_DIR}/distilbert_model/
  Threshold   : {DISTILBERT_THRESHOLD}

All files saved to: {DRIVE_STAGE2_DIR}
  distilbert_model/       ← load in Stage 3
  transformer_results.json
  loto_results.json
  fig8–fig13.png

Next: run Stage 3 notebook
""")

print("✓ Stage 2 complete")